# Signed URL downloads through Azure Front Door


Run cells in order. This notebook deploys billable resources into an existing base environment, then tests synthetic downloads without a client certificate. Edit [the configuration](signed-url.env.sh) first. Requires Azure CLI login, curl, jq, Node.js, GNU date and private Storage connectivity with blob data/delegation permissions. The generated azurefd.net hostname needs no DNS changes.

The public Premium profile and private Media account are reused, but the endpoint, origin group and non-anonymous container are separate. Existing routes and the Media root container are unchanged. Deployment needs AFD and Storage container management permissions. Stop on any failed assertion; use the cleanup cell with the same kernel variables if a later phase fails.

> **Cache caveat:** Storage validates SAS only on an origin request. A cached URL can remain usable after SAS expiry. UseQueryString separates SAS values; it does not enforce expiry at the edge. TTL starts when content is cached, not when the SAS is issued, and stale responses can also be served on origin errors. This is not a strict-expiry authorization design. Use synthetic data only. Do not print tokens or commit executed notebook outputs.


## Design details

The notebook deploys [a dedicated endpoint and route](../afd/signed-url.bicep) on the public Premium profile, using its generated `azurefd.net` hostname. No custom domain or DNS change is needed. The private Media storage account is reused with a separate, non-anonymous `signed-downloads` container. Existing endpoints, upload routes and the anonymous Media root container stay unchanged.

* [The download SAS helper](../scripts/new-download-sas.sh) issues blob-scoped, read-only, HTTPS-only user delegation SAS tokens. No account keys, client certificates or Front Door managed identity are used.
* The SAS signs a response Cache-Control override (`max-age=0, s-maxage=600` by default). The blob itself defaults to `private, no-store`.
* The route uses `UseQueryString`, so all SAS fields are part of the cache key. The same signed URL can reuse a cached response, a different SAS creates a separate entry.

The walkthrough checks unsigned requests, missing or tampered signatures, blob scope, read-only permissions, response content, cache hits and SAS expiry. After expiry, a never-warmed query variant must fail at Storage while the original cached URL can still succeed. Cache residency is not guaranteed: a missing cache hit is reported as an inconclusive cache demonstration, not as a passing authorization test.

Front Door queues successive configuration changes, so initial propagation can take about 30 minutes. The notebook polls for readiness and purge completion instead of accepting an ARM success as proof of data-plane availability. Run it from a host with private Storage connectivity (for example the demo VM). Cleanup purges only this run's two blob paths, deletes those blobs, unsets the tokens and removes the private temporary directory. The endpoint, origin group and empty container stay for reuse, so do not delete the shared resource group or the public profile to remove this demo.

Local tests: `node --test projects/mtlsupload/scripts/new-download-sas.test.cjs`. See [Front Door with Blob Storage](https://learn.microsoft.com/azure/frontdoor/scenario-storage-blobs#authentication) and [Front Door caching](https://learn.microsoft.com/azure/frontdoor/front-door-caching).


In [ ]:
source ./signed-url.env.sh &&
SIGNED_DIR=$(mktemp -d /tmp/afd-signed-url.XXXXXXXX) &&
export SIGNED_DIR &&
printf "Subscription: %s\nProfile: %s\nEndpoint: %s\nStorage: %s\nContainer: %s\nRun: %s\nTemporary files: %s\n" \
  "$SIGNED_SUB" "$SIGNED_PROFILE" "$SIGNED_ENDPOINT" "$SIGNED_STORAGE" "$SIGNED_CONTAINER" "$SIGNED_RUN_ID" "$SIGNED_DIR"

## Preflight and local tests

In [ ]:
(
  set -euo pipefail
  for tool in az curl jq node date; do command -v "$tool"; done
  node --test "$SIGNED_REPO_ROOT/projects/mtlsupload/scripts/new-download-sas.test.cjs"
  az account show --subscription "$SIGNED_SUB" --query '{subscription:id,state:state}' -o table
  [[ "$(az afd profile show --subscription "$SIGNED_SUB" -g "$SIGNED_RG" --profile-name "$SIGNED_PROFILE" --query sku.name -o tsv)" == Premium_AzureFrontDoor ]] || {
    echo 'Expected the existing Premium profile.' >&2; exit 1;
  }
  [[ "$(az storage account show --subscription "$SIGNED_SUB" -g "$SIGNED_RG" -n "$SIGNED_STORAGE" --query publicNetworkAccess -o tsv)" == Disabled ]] || {
    echo 'Expected private Media Storage; do not make the origin public for this demo.' >&2; exit 1;
  }
  az storage container list --subscription "$SIGNED_SUB" --account-name "$SIGNED_STORAGE" --auth-mode login --only-show-errors --query '[].name' -o table
  az bicep build --file "$SIGNED_TEMPLATE" --outfile "$SIGNED_DIR/template.json"
)


## Deploy the isolated endpoint and private container

In [ ]:
(
  set -euo pipefail
  az deployment group create --subscription "$SIGNED_SUB" -g "$SIGNED_RG" -n "$SIGNED_DEPLOYMENT" \
    --template-file "$SIGNED_TEMPLATE" --parameters profileName="$SIGNED_PROFILE" endpointName="$SIGNED_ENDPOINT" \
    storageName="$SIGNED_STORAGE" containerName="$SIGNED_CONTAINER" location="$SIGNED_LOCATION" --only-show-errors -o none
) &&
SIGNED_HOST=$(az deployment group show --subscription "$SIGNED_SUB" -g "$SIGNED_RG" -n "$SIGNED_DEPLOYMENT" --query properties.outputs.endpointHostName.value -o tsv) &&
[[ -n "$SIGNED_HOST" && "$SIGNED_HOST" != None ]] &&
SIGNED_URL="https://$SIGNED_HOST/$SIGNED_CONTAINER/$SIGNED_BLOB" &&
printf 'Endpoint: https://%s\nDownload path: /%s/%s\n' "$SIGNED_HOST" "$SIGNED_CONTAINER" "$SIGNED_BLOB"


## Approve only this demo's pending Private Link request

Front Door can reuse the public profile's existing Media private endpoint. If Azure creates another connection, only a pending request with this demo's exact request message is approved. Other pending connections are left alone. Readiness is checked against an actual Storage denial through Front Door, not just ARM deployment status; propagation can take several minutes.


In [ ]:
(
  set -euo pipefail
  STORAGE_ID=$(az storage account show --subscription "$SIGNED_SUB" -g "$SIGNED_RG" -n "$SIGNED_STORAGE" --query id -o tsv)
  READY=0
  for ((i=1; i<=SIGNED_READY_ATTEMPTS; i++)); do
    CONNECTIONS=$(az network private-endpoint-connection list --subscription "$SIGNED_SUB" --id "$STORAGE_ID" -o json)
    PENDING=$(jq -r '.[] | select(.properties.privateLinkServiceConnectionState.status == "Pending" and .properties.privateLinkServiceConnectionState.description == "afd-signed-url storage") | .id' <<< "$CONNECTIONS")
    while IFS= read -r connection; do
      [[ -z "$connection" ]] && continue
      az network private-endpoint-connection approve --subscription "$SIGNED_SUB" --id "$connection" --description 'approved for afd-signed-url storage' -o none
    done <<< "$PENDING"
    if STATUS=$(curl --silent --show-error --max-time 30 --dump-header "$SIGNED_DIR/ready.headers" --output "$SIGNED_DIR/ready.body" --write-out '%{http_code}' "$SIGNED_URL"); then
      printf 'Readiness attempt %s: HTTP %s\n' "$i" "$STATUS"
      if [[ "$STATUS" == 403 || "$STATUS" == 404 ]] && grep -qi '^x-ms-request-id:' "$SIGNED_DIR/ready.headers"; then
        READY=$((READY + 1))
        if (( READY >= 5 )); then
          echo 'Five consecutive origin responses confirmed; unsigned access is denied.'
          exit 0
        fi
      else
        READY=0
      fi
    else
      READY=0
      echo "Readiness attempt $i: transport/DNS failure; retrying." >&2
    fi
    sleep "$SIGNED_POLL_SECONDS"
  done
  echo 'Front Door did not reach the private origin in time. Inspect Private Link and route propagation.' >&2
  exit 1
)


## Seed two synthetic blobs with non-cacheable defaults

In [ ]:
(
  set -euo pipefail
  ACCESS=$(az storage container show --subscription "$SIGNED_SUB" --account-name "$SIGNED_STORAGE" -n "$SIGNED_CONTAINER" --auth-mode login --query properties.publicAccess -o tsv)
  [[ -z "$ACCESS" || "$ACCESS" == None ]] || { echo "Container is not private: $ACCESS" >&2; exit 1; }
  printf 'Synthetic signed download: %s\n' "$SIGNED_RUN_ID" > "$SIGNED_DIR/payload.txt"
  for blob in "$SIGNED_BLOB" "$SIGNED_OTHER_BLOB"; do
    az storage blob upload --subscription "$SIGNED_SUB" --account-name "$SIGNED_STORAGE" --container-name "$SIGNED_CONTAINER" \
      --name "$blob" --file "$SIGNED_DIR/payload.txt" --auth-mode login --overwrite false \
      --content-type text/plain --content-cache-control 'private, no-store' --only-show-errors -o none
  done
  az afd route show --subscription "$SIGNED_SUB" -g "$SIGNED_RG" --profile-name "$SIGNED_PROFILE" --endpoint-name "$SIGNED_ENDPOINT" \
    --route-name route-signed-url -o json | jq -e '.cacheConfiguration.queryStringCachingBehavior == "UseQueryString" and .linkToDefaultDomain == "Enabled"'
)


## Issue a read-only, blob-scoped SAS

The helper signs an HTTPS-only user delegation SAS using your Azure CLI identity, not an account key. Its signed response override sets `max-age=0` for browsers and `s-maxage=600` for Front Door by default, overriding the blob's non-cacheable default only for this SAS. The cache lifetime intentionally exceeds the short SAS validity so that the later test can expose the expiry caveat. The token stays in kernel memory and is never printed.


In [ ]:
SIGNED_SAS=$(DOWNLOAD_STORAGE="$SIGNED_STORAGE" DOWNLOAD_CONTAINER="$SIGNED_CONTAINER" AZURE_SUBSCRIPTION_ID="$SIGNED_SUB" \
  bash "$SIGNED_ISSUER" "$SIGNED_BLOB" "$SIGNED_SAS_MINUTES" "$SIGNED_CACHE_SECONDS") &&
SIGNED_SE=$(printf '%s' "$SIGNED_SAS" | tr '&' '\n' | sed -n 's/^se=//p' | sed 's/%3[Aa]/:/g') &&
[[ -n "$SIGNED_SE" ]] &&
SIGNED_EXPIRY=$(date -u -d "$SIGNED_SE" +%s) &&
printf 'SAS issued; expiry UTC: %s (signature hidden).\n' "$SIGNED_SE"


## HTTP assertions without exposing signed URLs

In [ ]:
signed_request() (
  set -euo pipefail
  label="$1"; expected="$2"; method="$3"; url="$4"; shift 4
  for ((attempt=1; attempt<=12; attempt++)); do
    status=$(curl --silent --show-error --max-time 60 --request "$method" \
      --dump-header "$SIGNED_DIR/$label.headers" --output "$SIGNED_DIR/$label.body" --write-out '%{http_code}' "$@" "$url")
    if [[ "$status" == 404 ]] && ! grep -qi '^x-ms-request-id:' "$SIGNED_DIR/$label.headers"; then
      echo "$label: Front Door routing is still propagating (generic 404), attempt $attempt/12." >&2
      sleep 5
      continue
    fi
    cache=$(awk 'tolower($1)=="x-cache:" {gsub("\r", ""); print $2}' "$SIGNED_DIR/$label.headers")
    error=$(awk 'tolower($1)=="x-ms-error-code:" {gsub("\r", ""); print $2}' "$SIGNED_DIR/$label.headers")
    printf '%s: HTTP %s, X-Cache=%s, StorageError=%s\n' "$label" "$status" "$cache" "$error"
    [[ ",$expected," == *",$status,"* ]] || { echo "$label: expected HTTP $expected, got $status." >&2; exit 1; }
    if [[ "$status" == 403 || "$status" == 404 ]]; then
      grep -qi '^x-ms-request-id:' "$SIGNED_DIR/$label.headers" || {
        echo "$label: denial did not come from Storage; authorization has not been verified." >&2; exit 1;
      }
    fi
    exit 0
  done
  echo "$label: no Storage response after bounded propagation retries." >&2
  exit 1
)
signed_assert_hit() (
  set -euo pipefail
  grep -Eqi '^x-cache: *(TCP_HIT|TCP_REMOTE_HIT)[[:space:]]*$' "$SIGNED_DIR/$1.headers" || {
    echo "$1: no confirmed cache hit. Cache residency is not guaranteed; the cache demonstration is inconclusive, not passing." >&2
    exit 1
  }
)


## Negative cases and first download

Anonymous requests to a private container can return 403 or 404. A tampered signature, a SAS for a different blob, and a write using this read-only SAS must fail with 403. These cases are checked before warming the valid URL. No client certificate is sent.


In [ ]:
(
  set -euo pipefail
  signed_request unsigned '403,404' GET "$SIGNED_URL"
  MISSING_SIG=$(printf '%s' "$SIGNED_SAS" | tr '&' '\n' | grep -v '^sig=' | paste -sd '&' -)
  signed_request missing-signature '403,404' GET "$SIGNED_URL?$MISSING_SIG"
  signed_request tampered-signature 403 GET "$SIGNED_URL?${SIGNED_SAS/sig=/sig=x}"
  signed_request different-blob 403 GET "https://$SIGNED_HOST/$SIGNED_CONTAINER/$SIGNED_OTHER_BLOB?$SIGNED_SAS"
  signed_request read-only 403 PUT "$SIGNED_URL?$SIGNED_SAS" \
    --header 'x-ms-blob-type: BlockBlob' --header 'x-ms-version: 2023-11-03' --data-binary 'must not be written'
  signed_request first-download 200 GET "$SIGNED_URL?$SIGNED_SAS"
  cmp "$SIGNED_DIR/payload.txt" "$SIGNED_DIR/first-download.body"
  grep -Eqi "^cache-control:.*s-maxage=$SIGNED_CACHE_SECONDS" "$SIGNED_DIR/first-download.headers"
  echo 'Downloaded content matches; the signed cache override reached the client.'
)


## Warm and prove a cache hit for exactly the same signed URL

In [ ]:
(
  set -euo pipefail
  for ((i=1; i<=12; i++)); do
    signed_request cached-download 200 GET "$SIGNED_URL?$SIGNED_SAS"
    cmp "$SIGNED_DIR/payload.txt" "$SIGNED_DIR/cached-download.body"
    if grep -Eqi '^x-cache: *(TCP_HIT|TCP_REMOTE_HIT)[[:space:]]*$' "$SIGNED_DIR/cached-download.headers"; then
      signed_assert_hit cached-download
      exit 0
    fi
    sleep 2
  done
  signed_assert_hit cached-download
)


## Expiry at Storage is not expiry at the edge

Wait until the signed expiry plus five seconds. A new, never-warmed query variant forces another cache key; Storage must reject its expired SAS. The original, already-cached URL can still succeed, which requires an actual `TCP_HIT` or `TCP_REMOTE_HIT` here. If the cached entry was evicted, the cell fails explicitly: do not interpret a miss as a demonstrated cache bypass or weaken the assertion.


In [ ]:
(
  set -euo pipefail
  WAIT=$((SIGNED_EXPIRY - $(date -u +%s) + 5))
  if (( WAIT > 0 )); then printf 'Waiting %s seconds for SAS expiry.\n' "$WAIT"; sleep "$WAIT"; fi
  signed_request expired-cold 403 GET "$SIGNED_URL?$SIGNED_SAS&demo=expired-$SIGNED_RUN_ID"
  signed_request expired-cached 200 GET "$SIGNED_URL?$SIGNED_SAS"
  signed_assert_hit expired-cached
  cmp "$SIGNED_DIR/payload.txt" "$SIGNED_DIR/expired-cached.body"
  echo 'Confirmed: Storage rejects the expired SAS on a miss, but the cached signed URL still downloads.'
)


## Purge the cached blob and verify origin rejection

Purge affects every query-string variant of this run's blob path. It is not instantaneous and is not an authorization mechanism. Poll until the same expired URL is rejected too; unlike a TTL wait, this explicitly removes the warmed entry.


In [ ]:
(
  set -euo pipefail
  az afd endpoint purge --subscription "$SIGNED_SUB" -g "$SIGNED_RG" --profile-name "$SIGNED_PROFILE" --endpoint-name "$SIGNED_ENDPOINT" \
    --content-paths "/$SIGNED_CONTAINER/$SIGNED_BLOB" --only-show-errors -o none
  for ((i=1; i<=SIGNED_READY_ATTEMPTS; i++)); do
    STATUS=$(curl --silent --show-error --max-time 60 --dump-header "$SIGNED_DIR/purged.headers" --output "$SIGNED_DIR/purged.body" --write-out '%{http_code}' "$SIGNED_URL?$SIGNED_SAS")
    printf 'Purge verification %s: HTTP %s\n' "$i" "$STATUS"
    if [[ "$STATUS" == 403 ]] && grep -qi '^x-ms-request-id:' "$SIGNED_DIR/purged.headers"; then
      echo 'The same expired URL is now rejected after purge.'
      exit 0
    fi
    [[ "$STATUS" == 200 ]] || { echo "Unexpected purge response: $STATUS" >&2; exit 1; }
    sleep "$SIGNED_POLL_SECONDS"
  done
  echo 'Purge did not remove the cached response within the verification window.' >&2
  exit 1
)


## Cleanup this run

Run this cell even after a failed test, using the same kernel variables. It purges only the two run-specific paths, deletes those blobs, removes private local response files and unsets the SAS. The endpoint, route, origin group, deployment record and empty container remain for reuse. Other blobs, Media content, mTLS uploads, DNS and profiles are not deleted. To remove the deployment later, delete only this dedicated endpoint, then its origin group and the empty signed-downloads container; never delete the shared profile or resource group.

References: [Storage behind Front Door](https://learn.microsoft.com/azure/frontdoor/scenario-storage-blobs#authentication), [Front Door caching](https://learn.microsoft.com/azure/frontdoor/front-door-caching).


In [ ]:
(
  set -euo pipefail
  [[ "$SIGNED_DIR" == /tmp/afd-signed-url.* && -d "$SIGNED_DIR" && -n "$SIGNED_RUN_ID" && "$SIGNED_BLOB" == "$SIGNED_RUN_ID.txt" && "$SIGNED_OTHER_BLOB" == "$SIGNED_RUN_ID-other.txt" ]] || {
    echo 'Cleanup scope is invalid; refusing to delete anything.' >&2; exit 1;
  }
  az afd endpoint purge --subscription "$SIGNED_SUB" -g "$SIGNED_RG" --profile-name "$SIGNED_PROFILE" --endpoint-name "$SIGNED_ENDPOINT" \
    --content-paths "/$SIGNED_CONTAINER/$SIGNED_BLOB" "/$SIGNED_CONTAINER/$SIGNED_OTHER_BLOB" --only-show-errors -o none
  for blob in "$SIGNED_BLOB" "$SIGNED_OTHER_BLOB"; do
    az storage blob delete --subscription "$SIGNED_SUB" --account-name "$SIGNED_STORAGE" --container-name "$SIGNED_CONTAINER" \
      --name "$blob" --auth-mode login --only-show-errors -o none
  done
  find "$SIGNED_DIR" -maxdepth 1 -type f -delete
  rmdir "$SIGNED_DIR"
  echo 'Synthetic blobs and local files removed; the dedicated endpoint remains deployed.'
) && unset SIGNED_SAS SIGNED_SE SIGNED_EXPIRY
